# Сравнение важности признаков по SHAP между финалистами

Смотрим, насколько согласована важность признаков между финалистами (логрег, лес, XGBoost, CatBoost) на двух наборах. Для каждой модели считаем SHAP, суммируем вклады one-hot уровней обратно к исходному признаку и берем топ (топ-5 для значимого набора из 10 признаков, топ-10 для no_collinear). Матрица присутствия показывает, какие признаки входят в топ у всех моделей, какие уникальны.

Отдельно для CatBoost сравниваем его SHAP-ранжирование со встроенной важностью (PredictionValuesChange): два разных способа измерить вклад должны давать близкий порядок. Логика в `src/shap_compare.py`, SHAP считается через `src/interpret.py` (нативный CatBoost обрабатывается корректно, без one-hot).

In [ ]:
import sys, pathlib, warnings
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))
warnings.filterwarnings('ignore')

from IPython.display import Image, Markdown, display
from src import shap_compare, config

shap_compare.run()
report = config.REPORTS_DIR / 'shap_compare.md'
display(Markdown(report.read_text(encoding='utf-8')))

## Матрицы присутствия и топы признаков

In [ ]:
for fset in shap_compare.FSETS:
    f = config.FIGURES_DIR / f'shap_overlap_{fset}.png'
    if f.exists():
        display(Image(str(f)))
for model in shap_compare.MODELS:
    for fset in shap_compare.FSETS:
        f = config.FIGURES_DIR / f'shapcmp_{model}_{fset}.png'
        if f.exists():
            display(Image(str(f)))

## CatBoost: SHAP против встроенной важности

In [ ]:
for fset in shap_compare.FSETS:
    f = config.FIGURES_DIR / f'shap_catboost_compare_{fset}.png'
    if f.exists():
        display(Image(str(f)))

## Вывод

Важность признаков согласована между финалистами. На значимом наборе в топ-5 у всех четырех моделей входят вид инсулинотерапии и стадия ХБП-С, на наборе no_collinear общими в топ-10 оказались вид инсулинотерапии, суточная доза инсулина, HbA1c, ЛПВП и глюкоза при поступлении. Сквозное ядро через все модели и наборы - вид инсулинотерапии и суточная доза инсулина, они в вершине у каждой модели. Это те же факторы, что значимы по Таблице 1: разные по устройству модели опираются на одни и те же клинические признаки.

Различия между моделями невелики и объяснимы: логистическая регрессия чуть выше ставит электролиты (натрий, калий), деревья и CatBoost - длительность СД, общий холестерин и тип СД. Это отражает разную природу моделей (линейные эффекты против взаимодействий), а не противоречие.

SHAP и встроенная важность CatBoost дают близкое ранжирование: Spearman 0.794 на значимом наборе (4 из 5 общих в топе) и 0.895 на no_collinear (9 из 10). Два независимых способа измерить вклад сходятся, что повышает доверие к интерпретации.

Итог: интерпретация устойчива и к выбору модели, и к способу измерения важности, и согласуется со статистикой Таблицы 1. Для сервиса это значит, что объяснение прогноза будет указывать на клинически осмысленные факторы независимо от того, какую из моделей выберет врач.